# Generate the YES / NO table of comorbidities

In [1]:
import pandas as pd
import numpy as np

# 1. Carregar dades
dfc_2 = pd.read_csv('COMORBIDITIES_100k.csv')

malalties_dict = {
    'Musculoskeletal Disorders': 0.4288,
    'Hypertension': 0.1864,
    'Arthrosis': 0.1323,
    'Asthma': 0.0893,
    'Diabetes': 0.0736,
    'Neoplasms': 0.0652,
    'Arthritis': 0.0646,
    'COPD': 0.0494,
    'Osteoporosis': 0.0437,
    'Chronic kidney disease': 0.0333,
    'Ischemic stroke': 0.0275,
    'Heart failure': 0.0240,
    'Dementia': 0.0233,
    'Cirrhosis': 0.0053,
    'AIDS': 0.0051
}

def generar_csv_comorbiditats(df):
    # Fem una còpia per no modificar l'original per accident
    df_res = df.copy()
    np.random.seed(42)
    
    noms_malalties = list(malalties_dict.keys())
    pesos_relatius = np.array(list(malalties_dict.values()))
    
    # Normalització robusta (per evitar errors de precisió decimal amb np.random.choice)
    pesos_relatius = pesos_relatius / pesos_relatius.sum()

    # Inicialitzem totes les columnes a 'No' de cop (més ràpid que un bucle)
    for m in noms_malalties:
        df_res[m] = 'No'

    # Iterem per omplir els 'Sí'
    # Nota: Amb 100k files, iterrows() pot trigar uns 15-20 segons.
    for i, row in df_res.iterrows():
        count = int(row['comorbidities'])
        
        if count > 0:
            # Seguretat: no podem triar més malalties de les que tenim al diccionari
            n_a_triar = min(count, len(noms_malalties))
            
            # Triem les malalties segons els pesos
            triades = np.random.choice(noms_malalties, size=n_a_triar, replace=False, p=pesos_relatius)
            
            # Useu .loc per assignar múltiples columnes a la vegada en aquella fila
            df_res.loc[i, triades] = 'Sí'
            
    return df_res

# Executar la funció
df_nou = generar_csv_comorbiditats(dfc_2)

# Guardar el resultat
df_nou.to_csv('comorb_detallades.csv', index=False)
print("El fitxer 'comorb_detallades.csv' s'ha creat correctament.")

El fitxer 'comorb_detallades.csv' s'ha creat correctament.


In [2]:
df_amb_comorb = df_nou[df_nou['comorbidities'] > 0]
columnes_malalties = [
    'Musculoskeletal Disorders', 'Hypertension', 'Arthrosis', 'Asthma', 
    'Diabetes', 'Neoplasms', 'Arthritis', 'COPD', 'Osteoporosis', 
    'Chronic kidney disease', 'Ischemic stroke', 'Heart failure', 
    'Dementia', 'Cirrhosis', 'AIDS'
]
frequencies = df_amb_comorb[columnes_malalties].apply(lambda x: (x == 'Sí').sum())

# Opcional: Percentatge respecte al total de persones amb alguna malaltia
percentatges = (frequencies / len(df_amb_comorb)) * 100
print("\nPercentatges:", percentatges.sort_values(ascending=False))


Percentatges: Musculoskeletal Disorders    45.844910
Hypertension                 22.288878
Arthrosis                    16.169908
Asthma                       11.238107
Diabetes                      9.116359
Neoplasms                     8.162836
Arthritis                     8.047066
COPD                          6.226320
Osteoporosis                  5.411720
Chronic kidney disease        4.169824
Ischemic stroke               3.452050
Heart failure                 2.959502
Dementia                      2.780584
Cirrhosis                     0.673571
AIDS                          0.654627
dtype: float64
